# Train a categorical factor graph

This example connects Gibbs sampling to a gradient for a two-node categorical model. The data favors matching categories. The model starts with zero interaction weights.

## Define the model and positive phase

A `CategoricalEBMFactor` assigns energy $-W[x_1,x_2]$ to each pair. The six observed pairs provide the positive phase. Each row is one complete state.

In [1]:
import equinox as eqx
import jax
import jax.numpy as jnp

from thrml import Block, BlockGibbsSpec, CategoricalNode, SamplingSchedule, sample_states
from thrml.factor import FactorSamplingProgram
from thrml.models import (
    CategoricalEBMFactor,
    CategoricalGibbsConditional,
    FactorizedEBM,
    contrastive_divergence_loss,
)

first, second = CategoricalNode(), CategoricalNode()
factor_blocks = [Block([first]), Block([second])]
state_blocks = [Block([first, second])]
weights = jnp.zeros((1, 3, 3))
model = FactorizedEBM([CategoricalEBMFactor(factor_blocks, weights)])
positive_samples = [jnp.array([[0, 0], [1, 1], [2, 2], [0, 0], [1, 1], [2, 2]], dtype=jnp.uint8)]

## Sample the negative phase

The two Gibbs blocks let the sampler update one node at a time. We read both nodes into one block, so the result has one sample axis and matches the positive phase.

In [2]:
spec = BlockGibbsSpec(factor_blocks, [])
samplers = [CategoricalGibbsConditional(3) for _ in spec.free_blocks]
program = FactorSamplingProgram(spec, samplers, model.factors, [])
initial_state = [jnp.array([0], dtype=jnp.uint8), jnp.array([0], dtype=jnp.uint8)]
schedule = SamplingSchedule(n_warmup=10, n_samples=64, steps_per_sample=1)
negative_samples = sample_states(jax.random.key(29), program, schedule, initial_state, [], state_blocks)
print(negative_samples[0].shape)

(64, 2)


## Estimate a gradient

`contrastive_divergence_loss` computes mean positive energy minus mean negative energy. Equinox differentiates the model weights while both sample phases stay fixed. The result is an energy-difference surrogate, not a normalized likelihood or a convergence metric.

In [3]:
def loss(current_model):
    return contrastive_divergence_loss(current_model, positive_samples, negative_samples, state_blocks)


value, gradient = eqx.filter_value_and_grad(loss)(model)
updates = jax.tree.map(lambda leaf: -0.1 * leaf, gradient)
updated_model = eqx.apply_updates(model, updates)
print(f"surrogate: {value:.3f}")
print(f"gradient norm: {jnp.linalg.norm(gradient.factors[0].weights):.3f}")
print(updated_model.factors[0].weights[0])

surrogate: 0.000
gradient norm: 0.519
[[ 0.02395833 -0.01875    -0.0140625 ]
 [-0.0125      0.02083333 -0.0078125 ]
 [-0.0109375  -0.0078125   0.02708334]]


## Check the update against exact probabilities

This two-node model has nine possible states. We can normalize all nine weights directly. The mean negative log likelihood of the observed pairs decreases after the update. This exact check is available because the example is small; the training gradient above used Gibbs samples.

In [4]:
def exact_mean_nll(current_model):
    current_weights = current_model.factors[0].weights[0]
    pairs = positive_samples[0]
    return -jnp.mean(current_weights[pairs[:, 0], pairs[:, 1]]) + jax.nn.logsumexp(current_weights)


nll_before = exact_mean_nll(model)
nll_after = exact_mean_nll(updated_model)
assert nll_after < nll_before
print(f"exact mean NLL: {nll_before:.3f} -> {nll_after:.3f}")

exact mean NLL: 2.197 -> 2.173


For another update, construct a sampling program from the new weights and draw a new negative phase. The positive and negative sample counts can differ. Flatten extra chain or batch axes before calling the loss.